In [ ]:
# -*- coding: utf-8 -*-
# ============================================================================
# CELLA 0 - IMPORTS & SETUP (eval epoca 2, v4 FIX)
# Batch session (Save & Run All / commit): niente idle timeout, 9h.
# FIX rispetto a v2/v3: il checkpoint epoca 2 viene copiato nella struttura
# lora_adapters/ che load_gemma4_duplex_checkpoint legge davvero.
# Ogni cella ha try/except che scrive l'errore su file (log kernel = 0 byte).
# ============================================================================
import os, sys, traceback
try:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Force single-GPU su Kaggle
    import json
    import time
    import gc
    import shutil
    import zipfile

    # Il dataset input puo' montare in piu' path: provali tutti
    _CANDIDATES = [
        "/kaggle/input/andrlab/gemma-4-omni-module",
        "/kaggle/input/datasets/andrlab/gemma-4-omni-module",
        "/kaggle/input/andrlab/gemma4-ep2-data",
        "/kaggle/input/datasets/andrlab/gemma4-ep2-data",
        "/kaggle/input/andrlab/gemma4-ep2-final-ckpt",
        "/kaggle/input/datasets/andrlab/gemma4-ep2-final-ckpt",
    ]
    _INPUT = None
    _CKPT_INPUT = None
    for c in _CANDIDATES:
        if os.path.isdir(c):
            if "gemma4-ep2-final-ckpt" in c:
                _CKPT_INPUT = c
            elif _INPUT is None:
                _INPUT = c
    print("Input dataset files:", os.listdir(_INPUT) if _INPUT else "MANCANTE")
    print("_INPUT =", _INPUT)
    print("_CKPT_INPUT =", _CKPT_INPUT)
    sys.stdout.flush()

    def _extract_zip(zpath, dst):
        with zipfile.ZipFile(zpath) as z:
            z.extractall(dst)

    def _copy_tree(src, dst):
        for item in os.listdir(src):
            s = os.path.join(src, item)
            d = os.path.join(dst, item)
            if os.path.islink(s):
                continue
            if os.path.isdir(s):
                os.makedirs(d, exist_ok=True)
                _copy_tree(s, d)
            else:
                os.makedirs(os.path.dirname(d), exist_ok=True)
                if not os.path.exists(d):
                    shutil.copy2(s, d)

    if _INPUT:
        # moduli python: copia diretta
        for f in ("kaggle_pipeline.py", "code2wav_decoder.py", "extract_qwen_subtalker.py",
                  "qwen_subtalker.py", "_epoca2.py"):
            s = os.path.join(_INPUT, f)
            if os.path.isfile(s):
                d = os.path.join("/kaggle/working", f)
                if not os.path.exists(d):
                    shutil.copy2(s, d)
        # dati: cartelle o zip
        for sub in ("data", "checkpoints", "code2wav_weights", "qwen_subtalker"):
            s = os.path.join(_INPUT, sub)
            z = os.path.join(_INPUT, sub + ".zip")
            if os.path.isdir(s):
                _copy_tree(s, sub)
            elif os.path.isfile(z):
                os.makedirs(sub, exist_ok=True)
                _extract_zip(z, sub)
                print(f"zip estratto: {sub}.zip -> {sub}/")
        # eventuale file_cleanup
        for f in os.listdir(_INPUT):
            if f.endswith(('.dat', '.log')):
                s = os.path.join(_INPUT, f)
                if not os.path.exists(os.path.join("/kaggle/working", f)):
                    shutil.copy2(s, "/kaggle/working")

    sys.path.insert(0, "/kaggle/working")
    import glob
    par = glob.glob("data/train_tokens_50.0h_2500s_mel.parquet")
    print("parquet 50h presente:", bool(par), par[:2])
    sys.stdout.flush()

    # ---- Checkpoint FINALE epoca 2 (FIX: copia in lora_adapters/, dove legge
    #      load_gemma4_duplex_checkpoint; NON a radice come in v2/v3) ----
    if _CKPT_INPUT:
        dst = "checkpoints/gemma4_duplex_sota_ckpt"
        os.makedirs(os.path.join(dst, "lora_adapters"), exist_ok=True)
        # struttura reale del dataset: può essere
        #   <input>/checkpoints/gemma4_duplex_sota_ckpt/...  (vecchio listing)
        #   <input>/gemma4_duplex_sota_ckpt/...              (nuovo, Kaggle appiattisce la singola sottodir)
        # oppure dentro una sottocartella extra dello zip.
        def _find_ckpt_root(root):
            for cand in (os.path.join(root, "checkpoints", "gemma4_duplex_sota_ckpt"),
                         os.path.join(root, "gemma4_duplex_sota_ckpt")):
                if os.path.isdir(cand):
                    return cand
            # ricerca ricorsiva limitata
            for dirpath, dirnames, filenames in os.walk(root):
                if os.path.basename(dirpath) == "gemma4_duplex_sota_ckpt" and "lora_adapters" in dirnames:
                    return dirpath
            return None
        base = _find_ckpt_root(_CKPT_INPUT)
        if base is None:
            for sub in os.listdir(_CKPT_INPUT):
                subp = os.path.join(_CKPT_INPUT, sub)
                if os.path.isdir(subp):
                    base = _find_ckpt_root(subp)
                    if base is not None:
                        break
        if base is None:
            print("!!! base ckpt NON trovata nel dataset ep2 — eval su checkpoint sbagliato!")
        else:
            for f in ("adapter_model.safetensors", "adapter_config.json", "README.md"):
                s = os.path.join(base, "lora_adapters", f)
                if os.path.isfile(s):
                    shutil.copy2(s, os.path.join(dst, "lora_adapters", f))
                    print(f"ckpt ep2: lora_adapters/{f} copiato")
            s = os.path.join(base, "custom_duplex_heads.pt")
            if os.path.isfile(s):
                shutil.copy2(s, os.path.join(dst, "custom_duplex_heads.pt"))
                print("ckpt ep2: custom_duplex_heads.pt copiato")
            import hashlib
            h = hashlib.md5(open(os.path.join(dst, "lora_adapters", "adapter_model.safetensors"), "rb").read()).hexdigest()[:8]
            print(f"ckpt ep2 md5(adapter): {h} (atteso 86a854e9 per il FINALE 24236)")
    else:
        print("WARNING: dataset gemma4-ep2-final-ckpt NON montato -> userò il ckpt del dataset principale (epoca 1)!")
    sys.stdout.flush()
    print("=== SETUP COMPLETATO ===")
    sys.stdout.flush()
except Exception:
    tb = traceback.format_exc()
    print("SETUP FALLITO:\n" + tb)
    sys.stdout.flush()
    with open("setup_ERROR.txt", "w", encoding="utf-8") as f:
        f.write(tb)
    raise


In [ ]:
# CELLA INSTALL - dipendenze Fase 2 su GPU P100 (sm_60)
# P100 non supporta torch cu128 (solo sm_70+). Serve torch cu118.
# Pattern VERIFICATO in test kernel: torch cu118 + --no-deps per non far toccare torch.
import subprocess, sys, os, re as _re

def gpu_info():
    try:
        r = subprocess.run([sys.executable, '-c',
            "import torch; print(torch.cuda.get_device_name(0)); print(torch.cuda.get_device_capability(0))"],
            capture_output=True, text=True, timeout=120)
        lines = [l for l in r.stdout.strip().splitlines() if l.strip()]
        cap = None
        if len(lines) > 1:
            m = _re.search(r'(\d+)', lines[1])
            cap = int(m.group(1)) if m else None
        return (lines[0] if lines else "?"), cap
    except Exception as e:
        return f"ERR {str(e)[:50]}", None

name, cap = gpu_info()
print("GPU rilevata:", name, "| compute capability:", cap)
sys.stdout.flush()

if cap is not None and cap < 7:
    # P100 (sm_60): stack cu118 (supporta Pascal), torch 2.6.0
    print(">>> GPU Pascal rilevata: installo torch 2.6.0+cu118 (supporta sm_60) <<<")
    sys.stdout.flush()
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet',
         '--index-url', 'https://download.pytorch.org/whl/cu118',
         'torch==2.6.0', 'torchvision==0.21.0', 'torchaudio==2.6.0'],
        capture_output=True, text=True, timeout=1800)
    print("torch cu118 rc:", r.returncode, (r.stdout + r.stderr)[-150:])
    sys.stdout.flush()
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps',
         'transformers==5.5.0', 'unsloth', 'unsloth_zoo', 'peft', 'safetensors', 'soundfile', 'pyarrow',
         'trl', 'xformers==0.0.29.post3', 'bitsandbytes', 'triton', 'tyro', 'structlog',
         'nest-asyncio', 'hf_transfer', 'diffusers', 'typer', 'sentencepiece', 'datasets'],
        capture_output=True, text=True, timeout=1800)
    print("stack cu118 rc:", r.returncode, (r.stdout + r.stderr)[-200:])
else:
    # T4 o superiore (cap>=7): stack normale cu128 (il kernel batch preinstalla torch cu128)
    # NOTA: qui NON usiamo --no-deps perche' le deps di transformers/unsloth (accelerate,
    # tokenizers, huggingface-hub, etc.) devono risolversi. torch resta quello preinstallato
    # (2.10+cu128) perche' transformers 5.5.0 richiede torch>=2.4 e unsloth torch<2.12.
    print(">>> GPU Volta+ rilevata: stack normale cu128 <<<")
    sys.stdout.flush()
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
         'transformers==5.5.0', 'unsloth', 'safetensors', 'soundfile', 'pyarrow'],
        capture_output=True, text=True, timeout=1800)
    print("stack cu128 rc:", r.returncode, (r.stdout + r.stderr)[-250:])
    sys.stdout.flush()

import importlib.metadata as md
for pkg in ['transformers', 'unsloth', 'peft', 'torch']:
    try: print(pkg + ": " + md.version(pkg))
    except Exception: print(pkg + ": MANCANTE")
# verifica GPU compatibile con torch
r2 = subprocess.run([sys.executable, '-c',
    "import torch; print('cuda ok:', torch.cuda.is_available(), torch.cuda.get_device_name(0))"],
    capture_output=True, text=True, timeout=120)
print("check cuda:", (r2.stdout + r2.stderr).strip()[-200:])


In [ ]:
# -*- coding: utf-8 -*-
"""
CELLA 2 - EVAL ESPLICITO sul checkpoint FINALE epoca 2 (step 24236, v4 FIX).
NON usa run_training_pipeline: validation e sintesi manuali, TUTTI i risultati
su file (data/val_ep2_final.txt, samples/ + metrics, synthesis_ERROR.txt).
Inoltre: applica la patch MCD (bug np.fft.dct) e verifica l'md5 del ckpt
caricato per PROVARE che è il finale epoca 2, non l'epoca 1.
"""
import os, sys, json, traceback, hashlib

try:
    sys.path.insert(0, "/kaggle/working")
    import torch
    import numpy as np
    import soundfile as sf
    from torch.utils.data import DataLoader

    import unsloth  # noqa: F401
    import kaggle_pipeline as KP

    # ---- PATCH MCD (FIX bug np.fft.dct -> DCT-2 numpy pura) ----
    PATCH = r'''
def _dct2_ortho(x):
    x = np.asarray(x, dtype=np.float64)
    N = x.shape[0]
    if N == 0:
        return x.copy()
    n = np.arange(N)
    k = np.arange(N)[:, None]
    _cos = np.cos((np.pi / N) * (n[None, :] + 0.5) * k)
    _y = _cos @ x
    _y[0] /= np.sqrt(2.0)
    _y *= np.sqrt(2.0 / N)
    return _y

def _mel_dist_mcd(a, b):
    n = min(a.shape[0], b.shape[0])
    if n == 0:
        return float("nan")
    _c = np.zeros_like(a[:n])
    _d = np.zeros_like(b[:n])
    for i in range(a.shape[1]):
        _c[:, i] = _dct2_ortho(a[:n, i])
        _d[:, i] = _dct2_ortho(b[:n, i])
    return float(np.sqrt(((_c[:, :13] - _d[:, :13]) ** 2).mean()) * (10.0 / np.log(10.0)))
'''
    exec(PATCH, {'np': np})
    # NB: exec() in un namespace dict NON esporta le funzioni nel namespace locale
    # (NameError in v4) -> estraiamo esplicitamente dal dict.
    _ns = {'np': np}
    exec(PATCH, _ns)
    KP._dct2_ortho = _ns['_dct2_ortho']
    KP._mel_dist_mcd = _ns['_mel_dist_mcd']
    print("PATCH MCD applicata (DCT-2 numpy pura)")
    sys.stdout.flush()

    # ---- 1) dati ----
    parquet_cache = f"data/train_tokens_{KP.DUPLEXCONV_MAX_HOURS}h_{KP.MDC_MAX_SAMPLES}s.parquet"
    KP.convert_parquet_to_memmap(parquet_cache)
    user_mel, agent_mel, agent_codes, text_by_window = KP.load_tokens_from_parquet(parquet_cache)
    print(f"agent frames: {agent_codes.shape[1]}")
    sys.stdout.flush()

    t_agent = agent_codes.shape[1]
    val_start = int((1.0 - KP.VAL_FRACTION) * t_agent)
    for e in KP.SEGMENT_ENDS:
        if e >= val_start:
            val_start = e
            break
    val_start = min(val_start, t_agent)
    n_val_w = max(0, (t_agent - val_start) // 64)
    print(f"VAL SPLIT: {val_start} train / {n_val_w} val finestre")
    sys.stdout.flush()

    def _slice_text(text_list, f_start):
        if text_list is None:
            return None
        return text_list[f_start // 64:]

    val_dataset = None
    if n_val_w > 0:
        val_dataset = KP.FullDuplexDataset(
            user_mel[val_start * KP.MEL_PER_AGENT_FRAME:],
            agent_mel[val_start * KP.MEL_PER_AGENT_FRAME:],
            agent_codes[:, val_start:],
            seq_len=64,
            text_by_window=_slice_text(text_by_window, val_start),
        )
    print(f"val_dataset: {len(val_dataset)} finestre" if val_dataset else "val_dataset: None")
    sys.stdout.flush()

    # ---- 2) modello + checkpoint epoca 2 ----
    duplex_model, _ = KP.load_gemma4_duplex_system()
    ckpt = "checkpoints/gemma4_duplex_sota_ckpt"
    KP.load_gemma4_duplex_checkpoint(duplex_model, ckpt)
    # PROVA: md5 dell'adapter caricato -> deve essere 86a854e9 (finale ep2 24236)
    h = hashlib.md5(
        open(os.path.join(ckpt, "lora_adapters", "adapter_model.safetensors"), "rb").read()
    ).hexdigest()[:8]
    print(f"checkpoint caricato: {ckpt} | md5(adapter)={h} (atteso 86a854e9)")
    sys.stdout.flush()
    with open("data/ckpt_md5.txt", "w", encoding="utf-8") as f:
        f.write(f"adapter_md5={h}\n")
    if h != "86a854e9":
        print("!!! WARNING: il checkpoint NON è il finale epoca 2 (24236) — risultati non validi per ep2 !!!")
        sys.stdout.flush()

    # ---- 3) VALIDATION esplicita ----
    base_device = next(duplex_model.base_model.parameters()).device
    duplex_model.eval()
    n_rc = 0
    for _vn, _vm in duplex_model.base_model.named_modules():
        if (("norm" in type(_vm).__name__.lower() or hasattr(_vm, "normalized_shape"))
                and getattr(_vm, "weight", None) is not None
                and _vm.weight.dtype != torch.float32):
            _vm.to(torch.float32)
            n_rc += 1
    if n_rc:
        print(f"LayerNorm ricastati fp32: {n_rc}")
        sys.stdout.flush()

    val_losses, val_txt = [], []
    if val_dataset is not None and len(val_dataset) > 0:
        vloader = DataLoader(val_dataset, batch_size=2, shuffle=False)
        with torch.inference_mode():
            for vb in vloader:
                try:
                    vout = duplex_model(
                        input_ids=vb["input_ids"].to(base_device),
                        attention_mask=vb["attention_mask"].to(base_device),
                        input_features=vb["input_features"].to(base_device),
                        input_features_mask=vb["input_features_mask"].to(base_device),
                        audio_targets=vb["audio_targets"].to(base_device),
                        frame_weights=vb["frame_weights"].to(base_device),
                        text_targets=vb["text_targets"].to(base_device),
                    )
                    val_losses.append(vout["loss"].item())
                    if "text_loss" in vout and vout["text_loss"] is not None:
                        val_txt.append(vout["text_loss"].item())
                except Exception as e:
                    print(f"val batch skip: {str(e)[:120]}")
                    sys.stdout.flush()

    val_loss = float(np.mean(val_losses)) if val_losses else float("nan")
    val_text = float(np.mean(val_txt)) if val_txt else float("nan")
    print(f">>> VAL_LOSS={val_loss:.4f} | VAL_TEXT={val_text:.4f} (su {len(val_losses)} batch)")
    sys.stdout.flush()

    os.makedirs("data", exist_ok=True)
    with open("data/val_ep2_final.txt", "w", encoding="utf-8") as f:
        f.write(f"val_loss={val_loss:.6f}\nval_text={val_text:.6f}\nn_val_batches={len(val_losses)}\n")
    print("val_ep2_final.txt scritto")

    # ---- 4) SINTESI esplicita con metrics su file ----
    try:
        synth_ds = val_dataset if (val_dataset is not None and len(val_dataset) > 0) else None
        if synth_ds is None:
            synth_ds = KP.FullDuplexDataset(
                user_mel[: 4096 * KP.MEL_PER_AGENT_FRAME],
                agent_mel[: 4096 * KP.MEL_PER_AGENT_FRAME],
                agent_codes[:, :4096],
                seq_len=64,
                text_by_window=_slice_text(text_by_window, 0),
            )
        generated = KP.synthesize_examples(duplex_model, synth_ds, n_windows=8)
        print(f"sintesi OK: {len(generated)} wav")
    except Exception as e:
        tb = traceback.format_exc()
        print(f"SINTESI FALLITA:\n{tb}")
        with open("synthesis_ERROR.txt", "w", encoding="utf-8") as f:
            f.write(tb)
    sys.stdout.flush()

    print("=== EVAL ESPLICITO EP2 v4 COMPLETATO ===")
    sys.stdout.flush()
except Exception:
    tb = traceback.format_exc()
    print("EVAL FALLITO:\n" + tb)
    sys.stdout.flush()
    with open("eval_ERROR.txt", "w", encoding="utf-8") as f:
        f.write(tb)
    raise
